### Day 34 PM Session | Week 6  PCA & Week 6 Comprehensive Review
---
| Part | Content |
|---|---|
| Part A | 12-Algorithm Reference + Wine Benchmark + Flowchart |
| Part B | Image Compression with PCA |
| Part C | Interview Questions |
| Part D | Week 6 Assessment Study Guide |


In [1]:
!pip install xgboost lightgbm -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, VotingClassifier, StackingClassifier,
                               ExtraTreesClassifier)
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy import stats
import xgboost as xgb
import lightgbm as lgb
import time


---
## Part A 12-Algorithm Week 6 Reference

In [ ]:
WEEK6_REFERENCE = {
    "Logistic Regression": {
        "description": "Linear classifier — models P(y|x) via sigmoid/softmax and log-loss.",
        "when":        "Linear boundary, need probabilities, large datasets, text.",
        "params":      {"C": "Inverse regularization strength", "solver": "lbfgs/saga for large n"},
        "code":        "LogisticRegression(C=1.0, max_iter=1000).fit(X_scaled, y)"
    },
    "Decision Tree": {
        "description": "Recursive binary splits on feature thresholds to minimize impurity.",
        "when":        "Need interpretable rules, mixed feature types, fast inference.",
        "params":      {"max_depth": "Depth limit prevents overfitting", "criterion": "gini or entropy"},
        "code":        "DecisionTreeClassifier(max_depth=5, criterion='gini').fit(X, y)"
    },
    "Random Forest": {
        "description": "Ensemble of decorrelated trees via bagging + random feature subsets.",
        "when":        "Tabular data, need feature importance, robust baseline.",
        "params":      {"n_estimators": "Number of trees (100+ typical)", "max_features": "sqrt default"},
        "code":        "RandomForestClassifier(n_estimators=100, n_jobs=-1).fit(X, y)"
    },
    "AdaBoost": {
        "description": "Sequential ensemble — upweights misclassified samples each round.",
        "when":        "Weak base learners, binary classification, clean data (outlier-sensitive).",
        "params":      {"n_estimators": "Number of weak learners", "learning_rate": "Contribution shrinkage"},
        "code":        "AdaBoostClassifier(n_estimators=100, learning_rate=1.0).fit(X, y)"
    },
    "XGBoost": {
        "description": "Gradient boosting with L1/L2 regularization and second-order gradients.",
        "when":        "Best accuracy on tabular, handles missing values, competitions.",
        "params":      {"n_estimators": "Trees to build", "max_depth": "3–10 typical"},
        "code":        "xgb.XGBClassifier(n_estimators=200, max_depth=6, verbosity=0).fit(X, y)"
    },
    "LightGBM": {
        "description": "Gradient boosting with leaf-wise growth and histogram-based splits.",
        "when":        "Large datasets (>100K), high-cardinality categoricals, speed needed.",
        "params":      {"num_leaves": "Controls complexity (31 default)", "learning_rate": "Step size"},
        "code":        "lgb.LGBMClassifier(num_leaves=31, learning_rate=0.05).fit(X, y)"
    },
    "Voting Classifier": {
        "description": "Combines diverse model predictions via majority vote or probability average.",
        "when":        "Multiple strong diverse models available, easy ensemble improvement.",
        "params":      {"voting": "hard (class) or soft (probability)", "estimators": "List of (name, model)"},
        "code":        "VotingClassifier(estimators=[('lr',lr),('rf',rf)], voting='soft').fit(X, y)"
    },
    "Stacking Classifier": {
        "description": "Meta-learner trained on out-of-fold predictions from base models.",
        "when":        "Maximum accuracy, diverse base models, time allows cross-validation.",
        "params":      {"estimators": "Base model list", "final_estimator": "Meta-learner (LR default)"},
        "code":        "StackingClassifier(estimators=[('svm',svm),('rf',rf)]).fit(X, y)"
    },
    "SVM (RBF)": {
        "description": "Maximum-margin classifier with RBF kernel for non-linear boundaries.",
        "when":        "High-dimensional, non-linear boundary, medium dataset, scaling applied.",
        "params":      {"C": "Margin hardness", "gamma": "Kernel width (scale default)"},
        "code":        "SVC(kernel='rbf', C=10, gamma='scale').fit(X_scaled, y)"
    },
    "KNN": {
        "description": "Classifies by majority vote among K nearest training points.",
        "when":        "Non-parametric baseline, small dataset, no training time budget.",
        "params":      {"n_neighbors": "K value (odd for binary)", "metric": "euclidean/cosine"},
        "code":        "KNeighborsClassifier(n_neighbors=5).fit(X_scaled, y)"
    },
    "K-Means": {
        "description": "Partitions data into K clusters minimizing within-cluster variance.",
        "when":        "Customer segmentation, unsupervised grouping, spherical cluster assumption.",
        "params":      {"n_clusters": "K (use elbow + silhouette)", "init": "k-means++ recommended"},
        "code":        "KMeans(n_clusters=3, init='k-means++', n_init=10).fit(X_scaled)"
    },
    "DBSCAN": {
        "description": "Density-based clustering — identifies core points and labels sparse regions as noise.",
        "when":        "Unknown K, non-spherical clusters, outlier detection needed.",
        "params":      {"eps": "Neighborhood radius", "min_samples": "Core point threshold"},
        "code":        "DBSCAN(eps=0.5, min_samples=5).fit_predict(X_scaled)"
    },
    "PCA": {
        "description": "Projects data onto eigenvectors of covariance matrix ordered by variance explained.",
        "when":        "Dimensionality reduction, visualization, remove multicollinearity.",
        "params":      {"n_components": "Dims to keep or float for variance (0.95)", "svd_solver": "auto"},
        "code":        "PCA(n_components=0.95).fit_transform(X_scaled)"
    }
}


def print_week6_reference(ref):
    for name, info in ref.items():
        print(f"\n{'='*62}")
        print(f"  {name}")
        print(f"{'='*62}")
        print(f"  DESCRIPTION : {info['description']}")
        print(f"  WHEN        : {info['when']}")
        print(f"  CODE        : {info['code']}")
        for pname, pdesc in info['params'].items():
            print(f"  PARAM [{pname}] : {pdesc}")


print_week6_reference(WEEK6_REFERENCE)


In [ ]:
def benchmark_on_wine(models_dict, cv_folds=5):
    wine = load_wine()
    X, y = wine.data, wine.target
    scaler = StandardScaler()
    X_s    = scaler.fit_transform(X)
    cv     = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

    records = []
    for name, model in models_dict.items():
        t0     = time.perf_counter()
        scores = cross_val_score(model, X_s, y, cv=cv, scoring='accuracy', n_jobs=-1)
        elapsed = time.perf_counter() - t0
        records.append({
            "Algorithm": name,
            "CV Mean":   round(scores.mean(), 4),
            "CV Std":    round(scores.std(), 4),
            "Time (s)":  round(elapsed, 3)
        })

    df = pd.DataFrame(records).sort_values("CV Mean", ascending=False).reset_index(drop=True)
    df.index += 1
    return df


wine_models = {
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost":             xgb.XGBClassifier(n_estimators=100, eval_metric='mlogloss',
                                              random_state=42, verbosity=0)
}

wine_df = benchmark_on_wine(wine_models)
print("Wine Dataset — 3-Algorithm Verification Benchmark")
print(wine_df.to_string())


### Algorithm Selection Flowchart

```
INPUT: Dataset Characteristics
       │
       ├── TASK = UNSUPERVISED?
       │     ├── Clusters spherical, K known?      → K-Means (n_init=10)
       │     ├── Non-spherical, outliers, K unknown?→ DBSCAN (tune eps)
       │     ├── Need merge hierarchy?              → Agglomerative (Ward)
       │     └── Reduce dimensions / visualize?     → PCA (n_components=0.95)
       │
       └── TASK = SUPERVISED?
             │
             ├── Features = TEXT / sparse?
             │     └── LinearSVC or LogisticRegression(L2)
             │
             ├── n_samples < 500?
             │     ├── n_features >> n_samples?  → SVM(Linear) / LR(L1)
             │     └── balanced dim?             → SVM(RBF) + GridSearchCV
             │
             ├── n_samples 500–10,000?
             │     ├── Need interpretability?    → Decision Tree (pruned)
             │     ├── Need probabilities?       → Logistic Regression
             │     ├── Non-linear, robust?       → Random Forest
             │     └── Best accuracy?            → XGBoost / LightGBM
             │
             └── n_samples > 10,000?
                   ├── Highest accuracy          → XGBoost / LightGBM
                   ├── Fast parallel baseline    → Random Forest
                   ├── Diverse ensemble ready?   → Stacking / Voting
                   └── Streaming data?           → SGDClassifier.partial_fit()
```


---
## Part B Image Compression with PCA

In [ ]:
def create_sample_image(size=128, seed=42):
    rng = np.random.RandomState(seed)
    img = np.zeros((size, size, 3), dtype=np.float64)
    for c in range(3):
        noise   = rng.rand(size, size)
        kernel  = np.outer(np.hanning(16), np.hanning(16))
        from scipy.signal import fftconvolve
        smooth  = fftconvolve(noise, kernel, mode='same')
        img[:, :, c] = (smooth - smooth.min()) / (smooth.max() - smooth.min())
    return img


def compress_channel_pca(channel, n_components):
    pca = PCA(n_components=n_components)
    compressed    = pca.fit_transform(channel)
    reconstructed = pca.inverse_transform(compressed)
    return reconstructed, pca.explained_variance_ratio_.sum()


def compress_image_pca(img, n_components):
    h, w, c = img.shape
    reconstructed = np.zeros_like(img)
    ev_sum = 0.0

    for ch in range(c):
        recon_ch, ev = compress_channel_pca(img[:, :, ch], n_components)
        reconstructed[:, :, ch] = recon_ch
        ev_sum += ev

    reconstructed = np.clip(reconstructed, 0, 1)
    mse  = float(np.mean((img - reconstructed) ** 2))
    psnr = 10 * np.log10(1.0 / mse) if mse > 1e-10 else float('inf')

    original_size    = h * w * c
    compressed_size  = n_components * (h + w) * c
    compression_ratio = original_size / compressed_size

    return reconstructed, mse, psnr, compression_ratio, ev_sum / c


def plot_pca_compression(img, components_list, figsize=(16, 8)):
    n = len(components_list) + 1
    fig, axes = plt.subplots(2, n, figsize=figsize)

    axes[0, 0].imshow(img)
    axes[0, 0].set_title("Original", fontsize=10, pad=5)
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')

    records = []
    for col, n_comp in enumerate(components_list, start=1):
        recon, mse, psnr, cr, ev = compress_image_pca(img, n_comp)
        axes[0, col].imshow(recon)
        axes[0, col].set_title(f"n={n_comp}
Ratio={cr:.1f}x", fontsize=9)
        axes[0, col].axis('off')

        axes[1, col].text(0.5, 0.65, f"MSE: {mse:.5f}",    ha='center', fontsize=9,
                          transform=axes[1, col].transAxes)
        axes[1, col].text(0.5, 0.35, f"PSNR: {psnr:.1f}dB", ha='center', fontsize=9,
                          transform=axes[1, col].transAxes)
        axes[1, col].text(0.5, 0.05, f"Var: {ev*100:.1f}%", ha='center', fontsize=8,
                          transform=axes[1, col].transAxes, color='gray')
        axes[1, col].axis('off')

        records.append({
            'n_components':   n_comp,
            'MSE':            round(mse, 6),
            'PSNR (dB)':      round(psnr, 2),
            'Compression':    f"{cr:.1f}x",
            'Variance (%)':   round(ev * 100, 2)
        })

    plt.suptitle("PCA Image Compression — Quality vs Components", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(records)


img = create_sample_image(size=128)
comp_list = [5, 20, 50, 100]
metrics_df = plot_pca_compression(img, comp_list)
print("\nCompression Metrics Summary:")
print(metrics_df.to_string(index=False))


In [ ]:
def plot_pca_scree(img_channel, title="Scree Plot", figsize=(10, 4)):
    pca_full = PCA().fit(img_channel)
    ev_ratio  = pca_full.explained_variance_ratio_
    cumulative = np.cumsum(ev_ratio)

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].plot(range(1, len(ev_ratio)+1), ev_ratio, 'o-', color='steelblue',
                 linewidth=1.5, markersize=3)
    axes[0].set_xlabel('Principal Component')
    axes[0].set_ylabel('Variance Explained')
    axes[0].set_title('Scree Plot')
    axes[0].set_xlim(0, min(50, len(ev_ratio)))
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(range(1, len(cumulative)+1), cumulative * 100, color='darkorange', linewidth=2)
    for threshold in [80, 90, 95, 99]:
        n_comp = np.argmax(cumulative >= threshold / 100) + 1
        axes[1].axhline(threshold, color='gray', linestyle='--', alpha=0.5)
        axes[1].text(len(cumulative) * 0.6, threshold + 0.5, f"{threshold}% @ n={n_comp}",
                     fontsize=8, color='gray')
    axes[1].set_xlabel('Principal Component')
    axes[1].set_ylabel('Cumulative Variance (%)')
    axes[1].set_title('Cumulative Explained Variance')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


plot_pca_scree(img[:, :, 0], title="Image Channel — PCA Scree Plot")


---
## Part C Interview Ready

### Q1 — Complete ML Pipeline: 1000 Samples, 200 Features

**Step 1 — EDA**
Check class balance, missing values, feature distributions, pairwise correlations. Identify obviously irrelevant or near-zero-variance features.

**Step 2 — Preprocessing**
```python
StandardScaler()     # mandatory before SVM, KNN, PCA, LR
SimpleImputer()      # fill missing values before scaling
```

**Step 3 — Dimensionality Reduction: PCA**
200 features with 1000 samples has p/n ratio of 0.2 — overfitting risk is real. PCA reduces noise and removes multicollinearity.
```python
PCA(n_components=0.95)    # typically collapses to 30–60 components
```
Chosen because: unsupervised, no label leakage, interpretable variance threshold.

**Step 4 — Baseline: Logistic Regression**
Fast, probabilistic, interpretable coefficients. L2 regularization handles residual multicollinearity. If accuracy is acceptable, deploy here.

**Step 5 — Stronger Model: Random Forest**
Non-linear boundaries. Feature importance confirms PCA retained discriminative variance. Robust to remaining outliers.

**Step 6 — Best Accuracy: XGBoost**
Use if Random Forest is insufficient. Regularized boosting squeezes out remaining signal. Tune `max_depth`, `learning_rate`, `subsample` via CV.

**Step 7 — Evaluation**
Stratified 5-fold CV. Report: accuracy, F1-macro (handles imbalance), AUC-ROC per class.

**Step 8 — Deployment**
Entire pipeline in `sklearn.Pipeline` → `joblib.dump`. Never fit the scaler or PCA on test data.


In [ ]:
def weekly_model_comparison(X, y, use_pca=False, pca_variance=0.95, cv_folds=5):
    if len(X) != len(y):
        raise ValueError("X and y must have the same number of samples")
    if not (0.0 < pca_variance <= 1.0):
        raise ValueError("pca_variance must be in (0, 1]")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    steps = [('scaler', StandardScaler())]
    if use_pca:
        steps.append(('pca', PCA(n_components=pca_variance)))

    preprocess = Pipeline(steps)
    X_train_t  = preprocess.fit_transform(X_train)
    X_test_t   = preprocess.transform(X_test)

    if use_pca:
        n_comp = preprocess.named_steps['pca'].n_components_
        print(f"PCA: {X_train.shape[1]} features → {n_comp} components "
              f"({pca_variance*100:.0f}% variance retained)")

    models = {
        "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
        "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost":             xgb.XGBClassifier(n_estimators=100, eval_metric='mlogloss',
                                                  random_state=42, verbosity=0),
        "SVM (RBF)":           SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
        "KNN":                 KNeighborsClassifier(n_neighbors=5)
    }

    cv      = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    records = []

    for name, model in models.items():
        t0      = time.perf_counter()
        scores  = cross_val_score(model, X_train_t, y_train,
                                  cv=cv, scoring='accuracy', n_jobs=-1)
        model.fit(X_train_t, y_train)
        test_acc = model.score(X_test_t, y_test)
        elapsed  = time.perf_counter() - t0

        records.append({
            "Model":    name,
            "CV Mean":  round(scores.mean(), 4),
            "CV Std":   round(scores.std(), 4),
            "Test Acc": round(test_acc, 4),
            "Gap":      round(abs(scores.mean() - test_acc), 4),
            "Time (s)": round(elapsed, 3)
        })

    df = pd.DataFrame(records).sort_values("CV Mean", ascending=False).reset_index(drop=True)
    df.index += 1
    return df


wine = load_wine()
print("Without PCA:")
df_no_pca = weekly_model_comparison(wine.data, wine.target, use_pca=False)
print(df_no_pca.to_string())

print("\nWith PCA (95% variance):")
df_pca = weekly_model_comparison(wine.data, wine.target, use_pca=True)
print(df_pca.to_string())


### Q3  Analyze: PCA 100→10 Features, Accuracy Drops 0.92→0.85

PCA retains 95% of **variance**, not 95% of **discriminative information**. These are fundamentally different objectives.

**Reason 1 — Discriminative signal is in low-variance directions**
PCA sorts components by variance in X, not by their correlation with y. A feature with low variance (e.g., a rare but highly specific biomarker) gets dropped even if it is the single best predictor of class membership. The 5% discarded variance may contain all the class boundary information.

**Reason 2 — Class-relevant axes are oblique to principal components**
The directions of maximum data spread rarely align with the directions that best separate class clusters. PCA is blind to labels — it has no knowledge of which variance is "useful" variance.

**Reason 3 — Hyperparameters miscalibrated in the new space**
Model hyperparameters tuned for 100D space no longer apply in 10D space. SVM's gamma controls kernel width relative to feature scale — a gamma optimized for 100 dimensions is too narrow or too wide in 10. KNN's K=5 may be optimal at 100D but poor at 10D where neighborhoods are denser.

**Fixes:**
```python
# Option 1: Supervised reduction — maximizes class separation directly
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis(n_components=n_classes-1)
X_lda = lda.fit_transform(X_train, y_train)

# Option 2: Feature selection by mutual information with y
from sklearn.feature_selection import SelectKBest, mutual_info_classif
selector = SelectKBest(mutual_info_classif, k=10)
X_selected = selector.fit_transform(X_train, y_train)

# Option 3: Re-tune hyperparameters in PCA space
param_grid = {'svm__C': [0.1,1,10], 'svm__gamma': [1e-3,0.01,0.1]}
GridSearchCV(Pipeline([('pca',pca),('svm',SVC())]), param_grid, cv=5)
```


---
## Part D  Week 6 Assessment Study Guide

---

### 1. Supervised Learning Quick Reference

| Algorithm | Objective | Key Params | Scaling? | Common Trap |
|---|---|---|---|---|
| Logistic Regression | Minimize log-loss | C, solver | Yes | Forget max_iter |
| Decision Tree | Minimize impurity (Gini/Entropy) | max_depth | No | No depth limit → overfit |
| Random Forest | Minimize SSE via bagging | n_estimators, max_features | No | Not tuning max_features |
| AdaBoost | Minimize weighted error | n_estimators, lr | No | Outliers destroy weights |
| XGBoost | Minimize regularized loss (2nd order) | max_depth, lr, subsample | No | No early_stopping → overfit |
| LightGBM | Leaf-wise boosting | num_leaves, lr | No | num_leaves too high |
| Voting | Aggregate predictions | voting | Depends | Weak base models |
| Stacking | Meta-learn on OOF preds | final_estimator | Depends | Leakage without CV stacking |
| SVM | Maximize margin | C, gamma, kernel | **Yes** | Missing StandardScaler |
| KNN | Nearest-neighbor vote | n_neighbors, metric | **Yes** | Curse of dimensionality |

---

### 2. Unsupervised Learning  Quick Reference

| Algorithm | Objective | Key Params | Scaling? | Common Trap |
|---|---|---|---|---|
| K-Means | Minimize within-cluster SSE | n_clusters, init | **Yes** | Wrong K, local minima |
| DBSCAN | Find dense connected regions | eps, min_samples | **Yes** | Not tuning eps per dataset |
| Agglomerative | Hierarchical merging | n_clusters, linkage | **Yes** | Memory O(n²) at large n |
| PCA | Maximize explained variance | n_components | **Yes** | Variance ≠ discriminative signal |

---

### 3. PCA Deep Dive

**Mathematics:**
```
1. Center: X_c = X - mean(X)
2. Covariance: Σ = X_cᵀ X_c / (n-1)
3. Eigendecompose: Σ = V Λ Vᵀ
4. Sort by eigenvalue (descending)
5. Project: X_reduced = X_c · V[:, :k]
```

**Practical code:**
```python
pca = PCA().fit(X_scaled)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_95   = np.argmax(cumvar >= 0.95) + 1

pca_fit = PCA(n_components=n_95).fit_transform(X_scaled)
```

---

### 4. Common Interview Questions

**Q: Bagging vs Boosting?**
Bagging: trains models in parallel on random subsets → reduces variance (RF). Boosting: trains sequentially, each model corrects previous errors → reduces bias (AdaBoost, XGBoost).

**Q: Why does SVM need scaling?**
RBF kernel computes `exp(-γ||x-z||²)`. Without scaling, large-magnitude features dominate distance calculations, making other features invisible.

**Q: Silhouette score interpretation?**
`s(i) = (b(i) - a(i)) / max(a(i), b(i))`. Near 1 = well-separated. Near 0 = on boundary. Negative = likely misclassified. Below 0.25 = no structure found.

**Q: When DBSCAN over K-Means?**
Unknown K, non-spherical clusters, outlier detection needed, varying cluster densities.

**Q: What is the kernel trick?**
Computes dot products in a high-dimensional feature space without explicitly computing the transformation: `K(x,z) = φ(x)·φ(z)`. RBF kernel implicitly maps to infinite dimensions.

---

### 5. Code Patterns to Know Cold

```python
# Full pipeline — never leaks test data
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=0.95)),
    ('model',  RandomForestClassifier())
])
cv_scores = cross_val_score(pipe, X, y, cv=StratifiedKFold(5))

# GridSearch on pipeline
param_grid = {'model__n_estimators': [100, 200]}
grid = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

# Clustering evaluation
km  = KMeans(n_clusters=3, init='k-means++', n_init=10).fit(X_s)
ari = adjusted_rand_score(y_true, km.labels_)
sil = silhouette_score(X_s, km.labels_)

# Elbow method
inertias = [KMeans(n_clusters=k, n_init=10).fit(X).inertia_ for k in range(2, 11)]

# PCA compression ratio
ratio = (H * W * C) / (n_components * (H + W) * C)
```

---
